In [1]:
# !pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu121
# !pip install torchtext==0.17.0
# !pip install numpy<2
# !pip install spacy
# !pip install portalocker>=2.0.0
# !pip install pywin32
# !pip install tensorflow

In [2]:
import torch.nn as nn
import torch
from torch.utils.data import DataLoader, TensorDataset
# from tensorflow.keras.datasets import imdb
# from tensorflow.keras.preprocessing.sequence import pad_sequences
import spacy
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from torchtext.vocab import build_vocab_from_iterator
import random


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\kevin\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\kevin\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\kevin\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\kevin\AppData\Roaming\Python\Python310\site-packages\traitlets\config\applicatio

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
import gzip

def read_gzip(path):
    with gzip.open(path, "rt", encoding="utf8") as f:
        return [line.strip() for line in f]
    
def load_data(path):

    train_en = read_gzip(f"{path}/train.en.gz")
    train_de = read_gzip(f"{path}/train.de.gz")

    val_en = read_gzip(f"{path}/val.en.gz")
    val_de = read_gzip(f"{path}/val.de.gz")

    test_en = read_gzip(f"{path}/test_2016_flickr.en.gz")
    test_de = read_gzip(f"{path}/test_2016_flickr.de.gz")

    train = list(zip(train_en, train_de))
    valid = list(zip(val_en, val_de))
    test = list(zip(test_en, test_de))

    return train, valid, test

In [5]:
train_data, valid_data, test_data = load_data("C:/Users/kevin/Desktop/RNN_ENGLISH_TO_GERMAN/multi30k-dataset/data/task1/raw")

In [6]:
en_train_seqs = [e for e,_ in train_data]
de_train_seqs = [d for _,d in train_data]

en_valid_seqs = [e for e,_ in valid_data]
de_valid_seqs = [d for _,d in valid_data]

en_test_seqs = [e for e,_ in test_data]
de_test_seqs = [d for _,d in test_data]

In [7]:
tokenizer_de = spacy.blank("de")
tokenizer_en = spacy.blank("en")

In [8]:
special_characters = ["<unk>","<pad>","<bos>","<eos>"]

In [9]:
def yield_list_of_en_tokens(seqs):
    for seq in seqs:
        yield [token.text for token in tokenizer_en(seq)]

def yield_list_of_de_tokens(seqs):
    for seq in seqs:
        yield [token.text for token in tokenizer_de(seq)]

# iterators build the vocabulary , without loading the entire dataset into the memory.

In [10]:


vocab_en = build_vocab_from_iterator( yield_list_of_en_tokens(en_train_seqs), special_first=True, specials=special_characters )
vocab_de = build_vocab_from_iterator( yield_list_of_de_tokens(de_train_seqs) , special_first=True, specials=special_characters )
vocab_en.set_default_index(vocab_en['<unk>'])
vocab_de.set_default_index(vocab_de['<unk>'])

In [11]:
def collate_fn(BATCH):

    en_seq_indices, de_seq_indices = [], []
    for en_list, de_list in BATCH:
        en_seq_indices.append( torch.tensor([vocab_en["<bos>"]] + en_list + [vocab_en["<eos>"]]).to(device) )
        de_seq_indices.append( torch.tensor([vocab_de["<bos>"]] + de_list + [vocab_de["<eos>"]]).to(device) )

    en_seq_indices = pad_sequence(en_seq_indices, padding_value=vocab_en["<pad>"],batch_first=True)
    de_seq_indices = pad_sequence(de_seq_indices, padding_value=vocab_de["<pad>"],batch_first=True)

    return en_seq_indices, de_seq_indices
    

In [12]:

class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self,en_list_tokens, de_list_tokens):
        self.en = en_list_tokens
        self.de = de_list_tokens
    
    def __len__(self):
        return len(self.en)
    
    def __getitem__(self, idx):
        return self.en[idx], self.de[idx]



In [13]:
def de_index_to_string(indices):
    converter = vocab_de.get_itos()
    return [converter[index] for index in indices]; 

def en_index_to_string(indices):
    converter = vocab_en.get_itos()
    return [converter[index] for index in indices]; 

In [14]:
BATCH_SIZE = 290//2
EPOCHS = 200
EN_VOCAB_SIZE = len(vocab_en)
DE_VOCAB_SIZE = len(vocab_de)

EMBEDDING_DIM = 128
HIDDEN_LAYER_SIZE = 1024
ENCODER_CONTEXT_VECTOR_SIZE = 1024
DECODER_CONTEXT_VECTOR_SIZE = 1024
TEACHER_FORCING_THRESHOLD = 0.5

In [15]:
# len(train_translation_dataset)

In [16]:
en_token_indices = [[vocab_en[token.text] for token in tokenizer_en(seq)] for seq in en_train_seqs]
de_token_indices = [[vocab_de[token.text] for token in tokenizer_de(seq)]  for seq in de_train_seqs] 

en_test_indices = [[vocab_en[token.text] for token in tokenizer_en(seq)] for seq in en_test_seqs]
de_test_indices = [[vocab_de[token.text] for token in tokenizer_de(seq)]  for seq in de_test_seqs] 

en_val_indices = [[vocab_en[token.text] for token in tokenizer_en(seq)] for seq in en_valid_seqs]
de_val_indices = [[vocab_de[token.text] for token in tokenizer_de(seq)]  for seq in de_valid_seqs] 



train_translation_dataset = TranslationDataset(en_token_indices , de_token_indices)
valid_translation_dataset = TranslationDataset(en_val_indices , de_val_indices)
test_translation_dataset = TranslationDataset(en_test_indices , de_test_indices)



train_loader = DataLoader(train_translation_dataset, collate_fn=collate_fn, batch_size=BATCH_SIZE, shuffle=True);
val_loader = DataLoader(valid_translation_dataset, collate_fn=collate_fn, batch_size=BATCH_SIZE, shuffle=True);
test_loader = DataLoader(test_translation_dataset, collate_fn=collate_fn, batch_size=BATCH_SIZE, shuffle=True);


In [ ]:
class RNN_ENCODER(nn.Module):
    # encoder's job is to encode the given input sentence into one vector.
    def __init__(self, num_embeddings, embedding_dim, context_vector_size, hidden_layer_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.fc1 = nn.Linear(embedding_dim + context_vector_size, hidden_layer_size)
        self.fc2 = nn.Linear(hidden_layer_size, context_vector_size)
        self.norm = nn.LayerNorm(context_vector_size) # normalization , rescales activations so they stay in a stable numerical range during training
        self.context_vector_size = context_vector_size

    def forward(self, batch_seq):
        batch_size, seq_len = batch_seq.shape
        context_vector = torch.zeros((batch_size, self.context_vector_size)).to(device)
        for i in range(seq_len):
            prev_context_vector = context_vector
            x = self.embedding(batch_seq[:, i])
            x = torch.tanh(torch.cat( (context_vector, x), dim = 1 ).to(device))
            x = torch.tanh(self.fc1(x))
            x = torch.tanh(self.fc2(x))
            context_vector = x
            context_vector = self.norm(prev_context_vector + context_vector)
            
        return context_vector



In [18]:
encoder = RNN_ENCODER(num_embeddings=EN_VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_layer_size=HIDDEN_LAYER_SIZE, context_vector_size=ENCODER_CONTEXT_VECTOR_SIZE).to(device)

In [ ]:
class RNN_DECODER(nn.Module):
    # Decoder's Job is to take the previous token_index and the context and generate the next token_index
    def __init__(self, context_vector_size, embedding_dim, hidden_layer_size, num_classes, num_embeddings):
        super().__init__();
        self.context_vector_size = context_vector_size
        self.embedding = nn.Embedding( num_embeddings=num_embeddings , embedding_dim = embedding_dim )
        self.fc1 = nn.Linear(context_vector_size , hidden_layer_size)
        self.fc2 = nn.Linear( hidden_layer_size, num_classes )
        self.norm = nn.LayerNorm(context_vector_size)
        # there are two kinds of normalizations, layer norm and batch norm, layer works best for RNNS
        # batch norm , normalizes by considering the variance and mean across the entire batch
        # where as layer norm, calculates variance and mean only for the current layer weights and therefore influences only the current sequence output.
        # thus layer norm is much more suitable for RNNS,LLMS and  Transformers
        # where as batch norm is used with CNNS. 
        self.fch1 = nn.Linear(context_vector_size + embedding_dim, hidden_layer_size)
        self.fch2 = nn.Linear( hidden_layer_size, context_vector_size )

    def forward(self, context_vector, prev_generated_token_vector):
        prev_context_vector = context_vector
        x = self.embedding(prev_generated_token_vector)
        x = torch.tanh(torch.cat(( context_vector, x ), dim=1).to(device))
        
        h = torch.tanh(self.fch1(x))
        h = torch.tanh(self.fch2(h))
        h = prev_context_vector + h # by adding previous context to the current context , we essentially create shortcut for the gradients to pass through.
        # without this connection, the gradients reach the previous hidden state weights, only via all of the operations, throught which the new state was acquired
        # but with this residual/skip connection, we essentially by pass all these connection and have direct access to the previous hidden state weights. 
        
        #Residual connections became famous after ResNet
        # showed they allowed 100+ layer networks to train stably.
        
        h = self.norm(h)
        
        o = torch.tanh(self.fc1(h))
        o = self.fc2(o)

        return o, h



In [20]:
decoder = RNN_DECODER(context_vector_size=DECODER_CONTEXT_VECTOR_SIZE, embedding_dim=EMBEDDING_DIM, hidden_layer_size=HIDDEN_LAYER_SIZE, num_classes=DE_VOCAB_SIZE, num_embeddings=DE_VOCAB_SIZE ).to(device)

In [21]:
for p in encoder.parameters():
    if p.dim() > 1:
        nn.init.orthogonal_(p)

for p in decoder.parameters():
    if p.dim() > 1:
        nn.init.orthogonal_(p)

In [22]:
# decoder.load_state_dict( torch.load("decoder.pt") )
# encoder.load_state_dict( torch.load("encoder.pt") )

In [25]:
def get_translation(en_indices):
    
    context_vectors = encoder(en_indices)
    # prev_token = torch.tensor(vocab_de["<bos>"]).to(device)
    end_token = vocab_de["<eos>"]
    # batch_size = en_indices.size(0)
    prev_token = torch.tensor([vocab_de["<bos>"]] ).to(device)

    german_tokens = []
    for i in range(30):
        if(prev_token.item() == end_token):
            break;
        else:
            preds, context_vectors = decoder(context_vector = context_vectors, prev_generated_token_vector = prev_token )
            de_token = preds.argmax(axis = 1)
            german_tokens.append(de_token)
            prev_token = de_token
    return " ".join(de_index_to_string(german_tokens))
            

Criterion = nn.CrossEntropyLoss(ignore_index=vocab_de["<pad>"] )
optimizer = torch.optim.Adam(list(decoder.parameters()) + list(encoder.parameters()), lr = 3e-4 )
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma = 0.75)

In [26]:

encoder.train()
decoder.train()

for epoch in range(EPOCHS):
    progress_bar = tqdm(train_loader)
    # at the start of every epoch the data points inside the dataloader are shuffled, one need not create a new dataloader, just to shuffle them.
    for batch in progress_bar:
        optimizer.zero_grad()
        en , de = batch
        context_vectors = encoder(en)
        batch_size, seq_len = de.shape
        total_loss = 0
        prev_token_indices = torch.stack( [ torch.tensor(vocab_de["<bos>"]) ] * BATCH_SIZE  ).to(device)
        
        for i in range(1, seq_len):
            
            # going frx`om index 1 , cause the index 0 is <bos> and it is already given as input to the model
            # meaning , the predicated value is for index 1, and the loss has to be calculated with index 1's value.
            preds, context_vectors = decoder(context_vector = context_vectors, prev_generated_token_vector = prev_token_indices )
            loss = Criterion(preds, de[:,i])
        
            total_loss += loss
            TFR = max(0.9 - (epoch * 0.005) , 0.3)
            if( random.random() < TFR ):
                prev_token_indices = de[:,i]
            else:
                prev_token_indices = preds.argmax(axis = 1).detach()
               
        avg_loss = total_loss / (seq_len - 1)
        avg_loss.backward()
        torch.nn.utils.clip_grad_norm_(  list(decoder.parameters()) + list(encoder.parameters()), max_norm=5 )
        optimizer.step()
        progress_bar.set_postfix(epoch = epoch, avg_loss = avg_loss.item(), lr = optimizer.param_groups[0]["lr"], TFR = TFR )
        
    with torch.no_grad():
        for en, de in val_loader:
            print("english seq : ", " ".join(en_index_to_string(en[0]))  )
            print("expected : ", " ".join(de_index_to_string(de[0])) )
            print("predicted : ", get_translation(en[0].unsqueeze(0)))
            break;

            
    scheduler.step()


100%|██████████| 200/200 [00:32<00:00,  6.24it/s, TFR=0.9, avg_loss=3.15, epoch=0, lr=0.0003]


english seq :  <bos> A man stare at a passing couple while walking down the block . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
expected :  <bos> Ein Mann starrt ein <unk> Paar an , während er den Häuserblock entlanggeht . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
predicted :  Ein Mann in einem blauen Hemd und einem blauen Oberteil und einem blauen Oberteil und einem rosafarbenen Oberteil , und einem grünen Hut , stellen her . <eos>


100%|██████████| 200/200 [00:37<00:00,  5.28it/s, TFR=0.895, avg_loss=3.22, epoch=1, lr=0.0003]


english seq :  <bos> A man in a white shirt and khaki pants crouches on a fallen tree trunk . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
expected :  <bos> Ein Mann in einem weißen T-Shirt und Khakihosen kauert auf einem umgestürzten Baumstamm . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
predicted :  Ein Mann in einem blauen Hemd und mit einem weißen Hemd und mit einem weißen Hemd , beide mit einem weißen Hemd , beide . <eos>


100%|██████████| 200/200 [00:38<00:00,  5.15it/s, TFR=0.89, avg_loss=2.88, epoch=2, lr=0.0003]


...

english seq :  <bos> A man wearing a bright , multi - color helmet is sitting on a motorcycle . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
expected :  <bos> Ein Mann mit einem leuchtend bunten Helm sitzt auf einem Motorrad . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
predicted :  Ein Mann in einem einem Hemd und mit einem . . <eos>


100%|██████████| 200/200 [00:34<00:00,  5.83it/s, TFR=0.445, avg_loss=4.06, epoch=91, lr=2.25e-5]


english seq :  <bos> A young child is standing alone on some jagged rocks . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
expected :  <bos> Ein kleines Kind steht allein auf einem zerklüfteten Felsen . <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
predicted :  Ein Mann in einem einem Hemd und mit einem . <eos>


 10%|▉         | 19/200 [00:03<00:29,  6.16it/s, TFR=0.44, avg_loss=4.46, epoch=92, lr=2.25e-5]


KeyboardInterrupt: 

In [ ]:
# print("---------------------")
# for name, param in encoder.named_parameters():
#     print(name, param.grad)
# print("---------------------")
print("decoder parameters : ")
for name, param in decoder.named_parameters():
    print(name, param.grad)
    break;
print("---------------------")

decoder parameters : 
embedding.weight tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.0003, -0.0002, -0.0001,  ...,  0.0003, -0.0011,  0.0002],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       device='cuda:0')
---------------------


In [ ]:
torch.save(encoder.state_dict(), "encoder.pt")
torch.save(decoder.state_dict(), "decoder.pt")